In [18]:
import pandas as pd
import glob
from utils import *

In [19]:
problems = [
    # (d, fine_m, p)
    (2, 11, 1),
    (2, 10, 3),
    (2, 9, 5),
    (3, 7, 1),
    (3, 6, 2),
    (3, 5, 3),
]

In [20]:
def get_problem_data(d, fine_m, p):
    dfs = []
    for file in glob.glob(f"../results/experiment*solvers*d{d}_p{p}_f{fine_m}*.csv"):
        dfs.append(pd.read_csv(file))
    return pd.concat(dfs, ignore_index=True)

In [21]:
for problem in problems:
    df = get_problem_data(*problem)
    print(
        problem,
        df[df["solver"] == "CUDSS"]["solve time"].min(),
        df[df["solver"] == "CUDSS"]["solver setup time"].min(),
    )

(2, 11, 1) 106.76326751708984 17051.427734375
(2, 10, 3) nan nan
(2, 9, 5) nan nan
(3, 7, 1) nan nan
(3, 6, 2) nan nan
(3, 5, 3) nan nan


In [22]:
def process_experiments_df(df: pd.DataFrame, fine_m: int) -> pd.DataFrame:
    if "solve time" not in df.columns:
        return pd.DataFrame()

    df = df[df.solver.str.contains("CG")].copy()
    df[["coarse m", "solvers m", "fine m"]] = df.apply(get_meshes, axis=1)

    summary = df.pivot_table(
        index=["solver"],
        columns=["coarse m", "solvers m"],
        values="solve time",
        aggfunc="min",
    )
    summary["best"] = summary.min(axis=1)

    best_amgx_key = summary["best"][summary.index.str.contains("AMGX")].idxmin()
    best_amgx_variant = best_amgx_key[8:].split(",")[0]
    best_amgx_time = summary["best"][best_amgx_key]

    best_asm_key = summary["best"][summary.index.str.contains("ASM")].idxmin()
    best_asm_time = summary["best"][best_asm_key]
    best_asm_precision = "fp32+fp16" if "float16" in best_asm_key else "fp32"

    best_by_meshes = summary[summary.index.str.contains("ASM")].min()
    best_by_meshes_prec = (
        summary[summary.index.str.contains("ASM")]
        .idxmin()
        .map(lambda s: "fp32+fp16" if not pd.isna(s) and "float16" in s else "fp32")
    )
    best_asm_meshing = best_by_meshes.idxmin()

    parallel_mask = [b == (fine_m, "S") for a, b in best_by_meshes.index]
    best_parallel_meshing = best_by_meshes[parallel_mask].idxmin()
    best_parallel_time = best_by_meshes[best_parallel_meshing]
    best_parallel_precision = best_by_meshes_prec[best_parallel_meshing]

    classical_mask = [a == b for a, b in best_by_meshes.index]
    best_classical_meshing = best_by_meshes[classical_mask].idxmin()
    best_classical_time = best_by_meshes[best_classical_meshing]
    best_classical_precision = best_by_meshes_prec[best_classical_meshing]

    return pd.Series(
        {
            "best_amgx_variant": best_amgx_variant,
            "best_amgx_time": best_amgx_time,
            "best_asm_meshing": best_asm_meshing,
            "best_asm_time": best_asm_time,
            "best_asm_precision": best_asm_precision,
            "best_parallel_meshing": best_parallel_meshing,
            "best_parallel_time": best_parallel_time,
            "best_parallel_precision": best_parallel_precision,
            "best_classical_meshing": best_classical_meshing,
            "best_classical_time": best_classical_time,
            "best_classical_precision": best_classical_precision,
        }
    )

In [23]:
def get_dofs(d, fine_m, p):
    element_dofs = {
        # (d, p)
        (2, 1): 3,
        (2, 3): 10,
        (2, 5): 21,
        (3, 1): 4,
        (3, 2): 10,
        (3, 3): 20,
    }
    return 2 ** (d * fine_m) * (2 if d == 2 else 6) * element_dofs[(d, p)]

In [24]:
def get_problem_summary(d, fine_m, p):
    df = get_problem_data(d, fine_m, p)
    if df.empty:
        return pd.Series()

    summary = process_experiments_df(df, fine_m)
    summary["d"] = d
    summary["m"] = fine_m
    summary["p"] = p
    summary["DoFs"] = get_dofs(d, fine_m, p)

    return summary

In [25]:
speedup_table = pd.DataFrame(get_problem_summary(*problem) for problem in problems)
speedup_table

/tmp/ipykernel_34554/1624179360.py:27: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  .idxmin()
/tmp/ipykernel_34554/1624179360.py:27: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  .idxmin()
/tmp/ipykernel_34554/1624179360.py:27: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  .idxmin()
/tmp/ipykernel_34554/1624179360.py:27: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  .idxmin()
/tmp/ipykernel_34554/1624179360.py:27: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future ve

,best_amgx_variant,best_amgx_time,best_asm_meshing,best_asm_time,best_asm_precision,best_parallel_meshing,best_parallel_time,best_parallel_precision,best_classical_meshing,best_classical_time,best_classical_precision,d,m,p,DoFs
0,L1_TRUNC,648.824036,"((9.0, C), (10.0, C))",725.783203,fp32+fp16,"((9.0, S), (11.0, S))",907.330994,fp32+fp16,"((9.0, S), (9.0, S))",783.312683,fp32+fp16,2,11,1,25165824
1,L1_TRUNC,1797.857910,"((9.0, S), (9.0, S))",1799.926270,fp32+fp16,"((9.0, S), (10.0, S))",1841.166138,fp32+fp16,"((9.0, S), (9.0, S))",1799.926270,fp32+fp16,2,10,3,20971520
2,L1_TRUNC,1792.261963,"((9.0, S), (9.0, S))",1854.992432,fp32+fp16,"((9.0, S), (9.0, S))",1854.992432,fp32+fp16,"((9.0, S), (9.0, S))",1854.992432,fp32+fp16,2,9,5,11010048
3,AGGRESIVE_L1,2308.794678,"((6.0, C), (7.0, C))",1830.571045,fp32+fp16,"((6.0, C), (7.0, S))",1862.314941,fp32+fp16,"((6.0, S), (6.0, S))",4187.350098,fp32+fp16,3,7,1,50331648
4,AGGRESSIVE_CHEB_L1_TRUNC,2229.360352,"((5.0, C), (6.0, C))",1557.173462,fp32+fp16,"((5.0, C), (6.0, S))",1629.627197,fp32+fp16,"((6.0, C), (6.0, C))",1904.449341,fp32+fp16,3,6,2,15728640
5,L1_TRUNC,1527.517822,"((5.0, C), (5.0, C))",862.405090,fp32+fp16,"((5.0, C), (5.0, S))",886.259888,fp32+fp16,"((5.0, C), (5.0, C))",862.405090,fp32+fp16,3,5,3,3932160


In [26]:
assert (speedup_table["best_asm_precision"] == "fp32+fp16").all()
assert (speedup_table["best_parallel_precision"] == "fp32+fp16").all()
assert (speedup_table["best_classical_precision"] == "fp32+fp16").all()

In [27]:
tab = pd.DataFrame()
tab[["$d$", "$p$"]] = speedup_table[["d", "p"]]
tab["$\\mathcal{T}_h$"] = speedup_table["m"].apply(
    lambda m: f"${format_mesh((m, 'S'))}$"
)
tab["DoFs"] = speedup_table["DoFs"].apply(lambda n: f"{n / 1_000_000:.1f}\\,{{M}}")

amgx_best = "best AmgX configuration found"
asm_best = "best ASM variant"

tab[(amgx_best, "name")] = speedup_table["best_amgx_variant"].map(
    lambda v: f"\\verb|{v.replace('AGGRESIVE', 'AGGRESSIVE')}|"
)
tab[(amgx_best, "time")] = speedup_table["best_amgx_time"] / 1000
tab[(asm_best, "$\\mathcal{T}_\\mathcal{H}$")] = speedup_table["best_asm_meshing"].map(
    lambda m: f"${format_mesh(m[0])}$"
)
tab[(asm_best, "$\\mathcal{T}_H$")] = speedup_table["best_asm_meshing"].map(
    lambda m: f"${format_mesh(m[1])}$"
)
tab[(asm_best, "time")] = speedup_table["best_asm_time"] / 1000
tab[(asm_best, "speedup")] = tab[(amgx_best, "time")] / tab[(asm_best, "time")]

tab.set_index(["$d$", "$p$", "$\\mathcal{T}_h$", "DoFs"], inplace=True)


def time_latex(t: float):
    return f"{t:.6f}\\, \\si{{\\second}}"


def speedup_latex(s: float):
    return f"{s:.6f} \\,\\(\\times\\)"


tab[(amgx_best, "time")] = tab[(amgx_best, "time")].apply(time_latex)
tab[(asm_best, "time")] = tab[(asm_best, "time")].apply(time_latex)
tab[(asm_best, "speedup")] = tab[(asm_best, "speedup")].apply(speedup_latex)

tab.columns = pd.MultiIndex.from_tuples(tab.columns)
tab.rename(columns=lambda c: f"{{{c}}}", inplace=True)
tab

{best AmgX configuration found}  \
                                                               {name}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                         
2   1   $\mathcal{S}_{11}$ 25.2\,{M}                  \verb|L1_TRUNC|   
    3   $\mathcal{S}_{10}$ 21.0\,{M}                  \verb|L1_TRUNC|   
    5   $\mathcal{S}_{9}$  11.0\,{M}                  \verb|L1_TRUNC|   
3   1   $\mathcal{S}_{7}$  50.3\,{M}             \verb|AGGRESSIVE_L1|   
    2   $\mathcal{S}_{6}$  15.7\,{M}  \verb|AGGRESSIVE_CHEB_L1_TRUNC|   
    3   $\mathcal{S}_{5}$  3.9\,{M}                   \verb|L1_TRUNC|   

                                                               \
                                                       {time}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                 
2   1   $\mathcal{S}_{11}$ 25.2\,{M}  0.648824\, \si{\second}   
    3   $\mathcal{S}_{10}$ 21.0\,{M}  1.797858\, \si{\second}   
    5   $\mathcal{S}_{9}$  11.0\,{M}  1.792262\, \si{\second}   
3   1   $\mathcal{S}_{7}$  50.3\,{M}  2.308795\, \si{\second}   
    2   $\mathcal{S}_{6}$  15.7\,{M}  2.229360\, \si{\second}   
    3   $\mathcal{S}_{5}$  3.9\,{M}   1.527518\, \si{\second}   

                                              {best ASM variant}  \
                                     {$\mathcal{T}_\mathcal{H}$}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                    
2   1   $\mathcal{S}_{11}$ 25.2\,{M}           $\mathcal{C}_{9}$   
    3   $\mathcal{S}_{10}$ 21.0\,{M}           $\mathcal{S}_{9}$   
    5   $\mathcal{S}_{9}$  11.0\,{M}           $\mathcal{S}_{9}$   
3   1   $\mathcal{S}_{7}$  50.3\,{M}           $\mathcal{C}_{6}$   
    2   $\mathcal{S}_{6}$  15.7\,{M}           $\mathcal{C}_{5}$   
    3   $\mathcal{S}_{5}$  3.9\,{M}            $\mathcal{C}_{5}$   

                                                          \
                                       {$\mathcal{T}_H$}   
$d$ $p$ $\mathcal{T}_h$    DoFs                            
2   1   $\mathcal{S}_{11}$ 25.2\,{M}  $\mathcal{C}_{10}$   
    3   $\mathcal{S}_{10}$ 21.0\,{M}   $\mathcal{S}_{9}$   
    5   $\mathcal{S}_{9}$  11.0\,{M}   $\mathcal{S}_{9}$   
3   1   $\mathcal{S}_{7}$  50.3\,{M}   $\mathcal{C}_{7}$   
    2   $\mathcal{S}_{6}$  15.7\,{M}   $\mathcal{C}_{6}$   
    3   $\mathcal{S}_{5}$  3.9\,{M}    $\mathcal{C}_{5}$   

                                                               \
                                                       {time}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                 
2   1   $\mathcal{S}_{11}$ 25.2\,{M}  0.725783\, \si{\second}   
    3   $\mathcal{S}_{10}$ 21.0\,{M}  1.799926\, \si{\second}   
    5   $\mathcal{S}_{9}$  11.0\,{M}  1.854992\, \si{\second}   
3   1   $\mathcal{S}_{7}$  50.3\,{M}  1.830571\, \si{\second}   
    2   $\mathcal{S}_{6}$  15.7\,{M}  1.557173\, \si{\second}   
    3   $\mathcal{S}_{5}$  3.9\,{M}   0.862405\, \si{\second}   

                                                             
                                                  {speedup}  
$d$ $p$ $\mathcal{T}_h$    DoFs                              
2   1   $\mathcal{S}_{11}$ 25.2\,{M}  0.893964 \,\(\times\)  
    3   $\mathcal{S}_{10}$ 21.0\,{M}  0.998851 \,\(\times\)  
    5   $\mathcal{S}_{9}$  11.0\,{M}  0.966183 \,\(\times\)  
3   1   $\mathcal{S}_{7}$  50.3\,{M}  1.261243 \,\(\times\)  
    2   $\mathcal{S}_{6}$  15.7\,{M}  1.431671 \,\(\times\)  
    3   $\mathcal{S}_{5}$  3.9\,{M}   1.771230 \,\(\times\)

In [28]:
time_col = "S[table-format=1.3\\,s, round-mode=places, round-precision=3]"
speedup_col = "S[table-format=1.3, round-mode=places, round-precision=3]"
dofs_col = "S[table-format=2.1\\,M]"
tab_latex = (
    tab.style.to_latex(
        hrules=True,
        multirow_align="t",
        multicol_align="c",
        column_format=f"rrc{dofs_col}|l{time_col}|cc{time_col}{speedup_col}",
    )
)

In [29]:
custom_header = r"""
 &  &  &  & \multicolumn{2}{c|}{{best AmgX configuration found}} & \multicolumn{4}{c}{{best ASM variant considered}} \\
$d$ & $p$ & $\mathcal{T}_h$ & {DoFs} & {name} & {time} & {$\mathcal{T}_\mathcal{H}$} & {$\mathcal{T}_H$} & {time} & {speedup} \\
"""

before_header = tab_latex.split("\\toprule")[0] + "\\toprule\n"
after_header = "\n\\midrule\n" + tab_latex.split("\\midrule")[1]
hacked_latex = before_header + custom_header + after_header
hacked_latex = hacked_latex.replace(
    "\\multirow[t]{3}{*}{3}", "\\midrule\n\\multirow[t]{3}{*}{3}"
)

In [30]:
with open("../docs/tables/experiment_speedups.tex", "w") as f:
    f.write(hacked_latex)

In [31]:
tab2 = pd.DataFrame()
tab2[["$d$", "$p$"]] = speedup_table[["d", "p"]]
tab2["$\\mathcal{T}_h$"] = speedup_table["m"].apply(
    lambda m: f"${format_mesh((m, 'S'))}$"
)
tab2["DoFs"] = speedup_table["DoFs"].apply(lambda n: f"{n / 1_000_000:.1f}\\,{{M}}")

asm_parallel = "best ASM variant with $\\mathcal{T}_H = \\mathcal{T}_h$"
asm_classical = "best ASM variant with $\\mathcal{T}_H = \\mathcal{T}_\\mathcal{H}$"

tab2[(asm_classical, "$\\mathcal{T}_\\mathcal{H}$")] = speedup_table[
    "best_classical_meshing"
].map(lambda m: f"${format_mesh(m[0])}$")
tab2[(asm_classical, "$\\mathcal{T}_H$")] = speedup_table["best_classical_meshing"].map(
    lambda m: f"${format_mesh(m[1])}$"
)
tab2[(asm_classical, "time")] = speedup_table["best_classical_time"] / 1000
tab2[(asm_classical, "speedup")] = (
    speedup_table["best_amgx_time"] / tab2[(asm_classical, "time")] / 1000
)

tab2[(asm_parallel, "$\\mathcal{T}_\\mathcal{H}$")] = speedup_table[
    "best_parallel_meshing"
].map(lambda m: f"${format_mesh(m[0])}$")
tab2[(asm_parallel, "$\\mathcal{T}_H$")] = speedup_table["best_parallel_meshing"].map(
    lambda m: f"${format_mesh(m[1])}$"
)
tab2[(asm_parallel, "time")] = speedup_table["best_parallel_time"] / 1000
tab2[(asm_parallel, "speedup")] = (
    speedup_table["best_amgx_time"] / tab2[(asm_parallel, "time")] / 1000
)

tab2.set_index(["$d$", "$p$", "$\\mathcal{T}_h$", "DoFs"], inplace=True)

tab2[(asm_classical, "time")] = tab2[(asm_classical, "time")].apply(time_latex)
tab2[(asm_parallel, "time")] = tab2[(asm_parallel, "time")].apply(time_latex)
tab2[(asm_classical, "speedup")] = tab2[(asm_classical, "speedup")].apply(speedup_latex)
tab2[(asm_parallel, "speedup")] = tab2[(asm_parallel, "speedup")].apply(speedup_latex)

tab2.columns = pd.MultiIndex.from_tuples(tab2.columns)
tab2.rename(columns=lambda c: f"{{{c}}}", inplace=True)
tab2

{best ASM variant with $\mathcal{T}_H = \mathcal{T}_\mathcal{H}$}  \
                                                                           {$\mathcal{T}_\mathcal{H}$}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                                                          
2   1   $\mathcal{S}_{11}$ 25.2\,{M}                                  $\mathcal{S}_{9}$                  
    3   $\mathcal{S}_{10}$ 21.0\,{M}                                  $\mathcal{S}_{9}$                  
    5   $\mathcal{S}_{9}$  11.0\,{M}                                  $\mathcal{S}_{9}$                  
3   1   $\mathcal{S}_{7}$  50.3\,{M}                                  $\mathcal{S}_{6}$                  
    2   $\mathcal{S}_{6}$  15.7\,{M}                                  $\mathcal{C}_{6}$                  
    3   $\mathcal{S}_{5}$  3.9\,{M}                                   $\mathcal{C}_{5}$                  

                                                         \
                                      {$\mathcal{T}_H$}   
$d$ $p$ $\mathcal{T}_h$    DoFs                           
2   1   $\mathcal{S}_{11}$ 25.2\,{M}  $\mathcal{S}_{9}$   
    3   $\mathcal{S}_{10}$ 21.0\,{M}  $\mathcal{S}_{9}$   
    5   $\mathcal{S}_{9}$  11.0\,{M}  $\mathcal{S}_{9}$   
3   1   $\mathcal{S}_{7}$  50.3\,{M}  $\mathcal{S}_{6}$   
    2   $\mathcal{S}_{6}$  15.7\,{M}  $\mathcal{C}_{6}$   
    3   $\mathcal{S}_{5}$  3.9\,{M}   $\mathcal{C}_{5}$   

                                                               \
                                                       {time}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                 
2   1   $\mathcal{S}_{11}$ 25.2\,{M}  0.783313\, \si{\second}   
    3   $\mathcal{S}_{10}$ 21.0\,{M}  1.799926\, \si{\second}   
    5   $\mathcal{S}_{9}$  11.0\,{M}  1.854992\, \si{\second}   
3   1   $\mathcal{S}_{7}$  50.3\,{M}  4.187350\, \si{\second}   
    2   $\mathcal{S}_{6}$  15.7\,{M}  1.904449\, \si{\second}   
    3   $\mathcal{S}_{5}$  3.9\,{M}   0.862405\, \si{\second}   

                                                             \
                                                  {speedup}   
$d$ $p$ $\mathcal{T}_h$    DoFs                               
2   1   $\mathcal{S}_{11}$ 25.2\,{M}  0.828308 \,\(\times\)   
    3   $\mathcal{S}_{10}$ 21.0\,{M}  0.998851 \,\(\times\)   
    5   $\mathcal{S}_{9}$  11.0\,{M}  0.966183 \,\(\times\)   
3   1   $\mathcal{S}_{7}$  50.3\,{M}  0.551374 \,\(\times\)   
    2   $\mathcal{S}_{6}$  15.7\,{M}  1.170606 \,\(\times\)   
    3   $\mathcal{S}_{5}$  3.9\,{M}   1.771230 \,\(\times\)   

                                     {best ASM variant with $\mathcal{T}_H = \mathcal{T}_h$}  \
                                                                 {$\mathcal{T}_\mathcal{H}$}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                                                
2   1   $\mathcal{S}_{11}$ 25.2\,{M}                                  $\mathcal{S}_{9}$        
    3   $\mathcal{S}_{10}$ 21.0\,{M}                                  $\mathcal{S}_{9}$        
    5   $\mathcal{S}_{9}$  11.0\,{M}                                  $\mathcal{S}_{9}$        
3   1   $\mathcal{S}_{7}$  50.3\,{M}                                  $\mathcal{C}_{6}$        
    2   $\mathcal{S}_{6}$  15.7\,{M}                                  $\mathcal{C}_{5}$        
    3   $\mathcal{S}_{5}$  3.9\,{M}                                   $\mathcal{C}_{5}$        

                                                          \
                                       {$\mathcal{T}_H$}   
$d$ $p$ $\mathcal{T}_h$    DoFs                            
2   1   $\mathcal{S}_{11}$ 25.2\,{M}  $\mathcal{S}_{11}$   
    3   $\mathcal{S}_{10}$ 21.0\,{M}  $\mathcal{S}_{10}$   
    5   $\mathcal{S}_{9}$  11.0\,{M}   $\mathcal{S}_{9}$   
3   1   $\mathcal{S}_{7}$  50.3\,{M}   $\mathcal{S}_{7}$   
    2   $\mathcal{S}_{6}$  15.7\,{M}   $\mathcal{S}_{6}$   
    3   $\mathcal{S}_{5}$  3.9\,{M}    $\mathcal{S}

In [32]:
tab_latex = (
    tab2.style.to_latex(
        hrules=True,
        multirow_align="t",
        multicol_align="c",
        column_format=f"rrc{dofs_col}|cc{time_col}{speedup_col}|cc{time_col}{speedup_col}",
    )
)

In [33]:
custom_header = r"""
 &  &  & & \multicolumn{4}{c|}{{best ASM variant with $\mathcal{T}_H = \mathcal{T}_\mathcal{H}$}} & \multicolumn{4}{c}{{best ASM variant with $\mathcal{T}_H = \mathcal{T}_h$}} \\
$d$ & $p$ & $\mathcal{T}_h$ & {DoFs} & {$\mathcal{T}_\mathcal{H}$} & {$\mathcal{T}_H$} & {time} & {speedup} & {$\mathcal{T}_\mathcal{H}$} & {$\mathcal{T}_H$} & {time} & {speedup} \\
"""
before_header = tab_latex.split("\\toprule")[0] + "\\toprule\n"
after_header = "\n\\midrule\n" + tab_latex.split("\\midrule")[1]
hacked_latex = before_header + custom_header + after_header
hacked_latex = hacked_latex.replace(
    "\\multirow[t]{3}{*}{3}", "\\midrule\n\\multirow[t]{3}{*}{3}"
)

In [34]:
with open("../docs/tables/experiment_speedups_2.tex", "w") as f:
    f.write(hacked_latex)